# 🌊 Coastal Assessment System — Project Overview

**Competition**: Geodata Systems Technologies · Coastal Assessment Challenge  
**Theme**: Smart Engineering for Sustainable Future through Innovation and Digitalisation

---

## 🎯 Project Goal

Build an **automated end-to-end pipeline** to classify Philippine coastal and marine features using Sentinel-2 satellite imagery and machine learning — producing pixel-level classification maps with measurable area statistics per ecosystem type.

### Classes Detected:
| # | Class | Ecological Significance |
|---|-------|------------------------|
| 1 | 🌿 **Seagrass** | Carbon sink & fish nursery — protected under Philippine Fisheries Code |
| 2 | 🏖️ **Sand** | Coastal boundary and erosion indicator |
| 3 | 🟤 **Seaweed** | Aquaculture indicator and biodiversity marker |
| 4 | 💧 **Water** | Open ocean and channel delineation |
| 5 | 🌲 **Landmass** | Terrestrial boundary, auto-generated from spectral thresholds |

---

## 🛰️ Data Source

**Copernicus Sentinel-2 Level 2A**
- 🆓 Free and open satellite data
- 📏 10m spatial resolution
- 🎨 13 spectral bands (we use 4)
- 📍 Philippine coastal study area

### Bands Used:
| Band | Name | Wavelength | Use Case |
|------|------|------------|----------|
| **B02** | Blue | 490 nm | Water body detection |
| **B03** | Green | 560 nm | NDWI water index |
| **B04** | Red | 665 nm | NDVI vegetation index |
| **B08** | NIR | 842 nm | Vegetation density + texture |

---

## 🧠 Technical Approach

### Phase 1: Feature Engineering (Notebook 02)
From 4 raw bands → 7-band composite:

1. **NDVI** = (NIR − Red) / (NIR + Red) → vegetation health
2. **NDWI** = (Green − NIR) / (Green + NIR) → water detection
3. **Texture** = local standard deviation of NIR in 3×3 window

### Phase 2: Spatial Feature Expansion (Notebook 03)
From 7 features → **21 features** per pixel:
- Center pixel values (7)
- 3×3 neighborhood mean per band (7)
- 3×3 neighborhood std per band (7)

This gives the model **spatial context** — each pixel is classified knowing what its neighbors look like, not just its own values.

### Phase 3: Model Training & Comparison (Notebooks 03 + 03b)

| Model | Accuracy | Notes |
|-------|----------|-------|
| RF Original | 88.07% | Baseline |
| RF + Balanced Weights | 87.61% | Class imbalance correction |
| RF + Deep Trees | 89.45% | 300 estimators, no depth limit |
| **RF + Spatial Features** | **92.66%** | ✅ Selected for production |
| RF + Grid Search | 88.53% | Hyperparameter tuning |
| 1D CNN | 88.07% | Comparison model |

**RF + Spatial Features** was selected: highest overall accuracy and wins on the ecologically critical Seagrass (F1: 0.943) and Seaweed (F1: 0.929) classes.

### Phase 4: Prediction & Reporting (Notebooks 04 + 05)
- Full image pixel-level classification using trained spatial model
- Area statistics in hectares per class
- Visualization maps and multi-year comparison framework

---

## 📁 Repository Structure

```
AI-ML-coastalAssessment/
├── notebooks/
│   ├── 00_overview.ipynb              ← You are here
│   ├── 01_data_loading.ipynb          ← Verify raw bands
│   ├── 02_preprocessing.ipynb         ← Build 7-band composite
│   ├── 03_model_training.ipynb        ← Train all RF variants, pick best
│   ├── 03b_cnn_comparison.ipynb       ← Train CNN, compare vs best RF
│   ├── 04_prediction_analysis.ipynb   ← Classify full image
│   └── 05_visualization_reports.ipynb ← Charts & area reports
├── data/
│   ├── raw/coastalImage/              ← B02.tiff B03.tiff B04.tiff B08.tiff
│   ├── aoi/                           ← explorer-aoi.geojson
│   ├── processed/                     ← processed_image_with_indices.tif
│   └── training/                      ← Ground_TruthFinal_with_landmass.csv
├── models/                            ← .pkl + scaler + metadata.json + .keras
├── outputs/                           ← Charts and visualizations
└── results/                           ← final_classification_map.tif + area_report.csv
```

**Run notebooks strictly in order: 00 → 01 → 02 → 03 → 03b → 04 → 05**


In [ ]:
# ============================================================
# ENVIRONMENT & DATA CHECK
# Run this before starting any other notebook
# ============================================================
import sys
import os
import importlib
import json

print("🔍 ENVIRONMENT & DATA CHECK")
print("=" * 65)

# ── Python version ──
print(f"\n🐍 Python: {sys.version.split()[0]}")
if sys.version_info < (3, 9):
    print("   ⚠️  Python 3.9+ recommended")
else:
    print("   ✅ Version OK")

# ── Required packages ──
print("\n📦 Required Packages:")
packages = {
    "numpy":      "numpy",
    "pandas":     "pandas",
    "matplotlib": "matplotlib",
    "rasterio":   "rasterio",
    "sklearn":    "scikit-learn",
    "scipy":      "scipy",
    "geopandas":  "geopandas",
    "joblib":     "joblib",
    "tensorflow": "tensorflow",
}

all_ok = True
for pkg, pip_name in packages.items():
    try:
        module = importlib.import_module(pkg)
        version = getattr(module, "__version__", "unknown")
        print(f"   ✅ {pkg:<15} {version}")
    except ImportError:
        print(f"   ❌ {pkg:<15} NOT INSTALLED → pip install {pip_name}")
        all_ok = False

# ── Raw data files ──
print("\n📡 Raw Satellite Bands:")
band_dir = "../data/raw/coastalImage"
required_bands = ["B02.tiff", "B03.tiff", "B04.tiff", "B08.tiff"]
if os.path.exists(band_dir):
    found = os.listdir(band_dir)
    for band in required_bands:
        if band in found:
            print(f"   ✅ {band}")
        else:
            print(f"   ❌ {band} — MISSING")
            all_ok = False
else:
    print(f"   ❌ {band_dir} folder NOT FOUND")
    all_ok = False

# ── AOI ──
print("\n🗺️  Area of Interest:")
aoi_dir = "../data/aoi"
if os.path.exists(aoi_dir):
    geojsons = [f for f in os.listdir(aoi_dir) if f.endswith(".geojson")]
    if geojsons:
        print(f"   ✅ {geojsons[0]}")
    else:
        print("   ⚠️  No .geojson found — pipeline will skip AOI clipping")
else:
    print("   ⚠️  ../data/aoi/ not found — pipeline will skip AOI clipping")

# ── Training data ──
print("\n🎯 Training Data:")
csv_candidates = [
    "../data/training/Ground_TruthFinal_with_landmass.csv",
    "../data/training/trainingdata/Ground_TruthFinal_with_landmass.csv",
    "../data/training/Ground_TruthFinal.csv",
]
csv_found = False
for csv_path in csv_candidates:
    if os.path.exists(csv_path):
        import pandas as pd
        df = pd.read_csv(csv_path)
        classes = sorted(df["Value"].dropna().unique().astype(int).tolist())
        print(f"   ✅ {os.path.basename(csv_path)}")
        print(f"      Samples: {len(df)} | Classes: {classes}")
        csv_found = True
        break
if not csv_found:
    print("   ⚠️  No training CSV found — notebook 03 will generate landmass points")

# ── Processed image ──
print("\n🖼️  Processed Image:")
proc_path = "../data/processed/processed_image_with_indices.tif"
if os.path.exists(proc_path):
    import rasterio
    with rasterio.open(proc_path) as src:
        print(f"   ✅ processed_image_with_indices.tif")
        print(f"      Bands: {src.count} | Size: {src.width}×{src.height} px | CRS: EPSG:{src.crs.to_epsg()}")
else:
    print("   ⚠️  Not yet generated — run notebook 02 first")

# ── Trained model ──
print("\n🤖 Trained Model:")
model_path = "../models/coastal_classifier_model.pkl"
meta_path  = "../models/model_metadata.json"
if os.path.exists(model_path) and os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    print(f"   ✅ {meta.get('model_type', 'RF model')}")
    print(f"      Accuracy: {meta.get('test_accuracy', 0)*100:.2f}% | "
          f"Features: {meta.get('n_features', '?')} | "
          f"Classes: {meta.get('classes', [])}")
else:
    print("   ⚠️  Not yet trained — run notebooks 03 + 03b first")

# ── CNN model ──
print("\n🧠 CNN Model:")
cnn_path = "../models/cnn_coastal_classifier.keras"
if os.path.exists(cnn_path):
    print(f"   ✅ cnn_coastal_classifier.keras")
else:
    print("   ⚠️  Not yet trained — run notebook 03b first")

# ── Output directories ──
print("\n📂 Output Directories:")
for d in ["../outputs", "../results", "../models"]:
    os.makedirs(d, exist_ok=True)
    print(f"   ✅ {d}/")

# ── Final summary ──
print("\n" + "=" * 65)
if all_ok:
    print("✅ ALL CRITICAL CHECKS PASSED — ready to run the pipeline")
else:
    print("⚠️  SOME CHECKS FAILED — resolve issues above before continuing")
print("\n📌 Run order: 01 → 02 → 03 → 03b → 04 → 05")


---

## 🎓 For Team Members

### Run Order:

| Step | Notebook | What it does | Key Output |
|------|----------|-------------|------------|
| 1 | `01_data_loading.ipynb` | Verify raw bands, visualize | `outputs/01_spectral_bands.png` |
| 2 | `02_preprocessing.ipynb` | Build 7-band composite + AOI clip | `data/processed/processed_image_with_indices.tif` |
| 3 | `03_model_training.ipynb` | Train all RF variants, pick best, save | `models/coastal_classifier_model.pkl` |
| 4 | `03b_cnn_comparison.ipynb` | Train 1D CNN, compare vs best RF | `models/cnn_coastal_classifier.keras` |
| 5 | `04_prediction_analysis.ipynb` | Classify full image using best RF | `results/final_classification_map.tif` |
| 6 | `05_visualization_reports.ipynb` | Area stats, charts, trend analysis | `results/final_area_report.csv` |

---

### Common Issues & Fixes:

❌ **`ModuleNotFoundError: No module named 'tensorflow'`**  
→ `pip install tensorflow` then restart the kernel

❌ **`FileNotFoundError: processed_image_with_indices.tif`**  
→ Run notebook 02 first — this file is generated there

❌ **`ValueError: X has 7 features but StandardScaler expects 21`**  
→ The spatial model (21 features) is active. Re-run notebook 03 fully so the spatial scaler is saved correctly

❌ **`NameError: name 'best_model' is not defined`**  
→ Kernel was reset — do **Kernel → Restart & Run All** in notebook 03

❌ **`NameError: name 'results' is not defined`** (in 03b)  
→ `results` no longer exists in 03b — it lives in notebook 03. Re-run 03b Cell 1 first

❌ **`rasterio CRS mismatch warning`**  
→ Not an error — the pipeline handles CRS reprojection automatically

---

## 📊 Final Model Performance

### RF + Spatial Features (Production Model):
| Metric | Score |
|--------|-------|
| **Overall Accuracy** | **92.66%** |
| Precision | 0.9263 |
| Recall | 0.9263 |
| F1-Score | 0.9263 |
| Training samples | ~871 |
| Features per pixel | 21 |

### Per-Class F1 — Best RF vs 1D CNN:
| Class | RF F1 | CNN F1 | Winner |
|-------|-------|--------|--------|
| 🌿 Seagrass | **0.943** | 0.917 | ✅ RF |
| 🏖️ Sand | 0.909 | **0.938** | CNN |
| 🟤 Seaweed | **0.929** | 0.906 | ✅ RF |
| 💧 Water | 0.963 | **1.000** | CNN |
| 🌲 Landmass | 0.769 | **0.875** | CNN |

RF selected for production — wins on Seagrass and Seaweed, the classes protected under the Philippine Fisheries Code (RA 8550).

---

## 🌎 Real-World Impact

- 🌿 **Seagrass mapping** → identifies carbon sink zones (~83g C/m²/yr) and fish nursery areas
- 🟤 **Seaweed distribution** → guides sustainable aquaculture placement
- 🏖️ **Sand/Water boundary** → enables coastal erosion tracking over time
- 🌲 **Landmass auto-detection** → no manual labeling needed, uses NDVI + NDWI thresholds
- 📊 **Quantified area statistics** → evidence-based data for policy decisions

### Note on Multi-Year Analysis:
Notebook 05 includes a full multi-year comparison framework. However, Copernicus had a platform-side access issue that prevented retrieval of specific spectral bands from earlier timestamps during this project period. The pipeline is built and ready — once data becomes accessible, no code changes are needed.

---

## 📌 Quick Start

```bash
# 1. Install dependencies
pip install -r requirements.txt

# 2. Add Sentinel-2 bands to:
#    data/raw/coastalImage/  →  B02.tiff  B03.tiff  B04.tiff  B08.tiff

# 3. Run notebooks in order starting from 01
```

**Questions?** Check [README.md](../README.md) or ask your team lead.

---
*Smart Engineering for Sustainable Future through Innovation and Digitalisation — WED 2026*
